# 7. LightGBM con encoding ancho y artefactos del generador

Cambio de enfoque. Los notebooks 5 y 6 codifican **una variable con mucho cuidado** (ingreso, en dos
etapas, con corrección de Newton jerárquica) y llegaron a **0,945468 OOF / 0,94573 público**. Las dos
últimas iteraciones rindieron +0,0003 cada una.

La alternativa es la opuesta: **codificar todo, a varias granularidades, con un modelo simple**. Es lo
que hace la solución pública de referencia, que con un LightGBM directo llega a CV 0,94607 — por
encima de nuestro pipeline de dos etapas, y a una fracción del cómputo.

**Este notebook no busca ganarle al notebook 6 por sí solo.** Busca un modelo *distinto*, porque todos
los intentos de ensamble del episodio fracasaron por correlación: logit 0,9889, XGBoost tuneado
0,9945, híbrido 0,9984. Si este modelo queda por debajo pero decorrelacionado, sirve igual.

## Lo que se incorpora, y por qué

| Elemento | Evidencia |
|---|---|
| **Target encoding triple** (`smooth` = auto / 10 / 100) sobre 15 claves | Es la diferencia estructural con nuestro enfoque |
| **Claves multi-escala** de ingreso y distancia (exacto, /100, /1000) | Réplica del patrón que ya funcionó en el notebook 5 |
| **Dígitos de `Daily_Commute_km`** (unidades y primer decimal) | Medido: residuo por dígito con z hasta 3,9 contra el OOF del notebook 6 |
| **Flag `ingreso >= 170537`** | Medido: 393 filas en train, tasa **1,0000**, el notebook 6 predice 0,964 |
| **Flag `ingreso == 30000`** | Mode collapse del generador; ya lo capturamos, entra por completitud |
| **Frecuencias sobre train+test** | Transductivo y legítimo: no usa etiquetas. Hasta ahora sólo usábamos train |

**Descartado por medición previa:** los dígitos de `Annual_Income_USD` (ratio de varianza residual
0,2-1,0, o sea nada) y el dataset original como fuente de codificación — 8.571 de sus 9.093 valores de
ingreso aparecen **una sola vez**, da AUC 0,516 por sí solo y correlaciona +0,0089 con nuestro
residuo. La competencia ya tiene ~50 filas por valor de ingreso; el original tiene 1.

**Advertencia sobre el tamaño esperable.** Los efectos de dígitos son reales pero chicos: el rango del
residuo entre dígitos es 0,0096 en probabilidad, contra el sd de 0,10 que tenía el efecto por valor de
ingreso. Son veinte veces menores. Y el flag del cliff afecta 156 filas de test.

> **Tiempo: ~30 minutos.** En una corrida de prueba el encoding tardó 38 s y el fit 45 s por fold.
> Acá el encoding además transforma el test (286k filas) tres veces por fold, así que sube a ~60 s, y
> son cinco variantes que comparten ese encoding.

**Métrica:** ROC AUC · **Dataset:** Playground Series S6E9

In [1]:
import time, json
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata, ttest_rel
import warnings
warnings.filterwarnings("ignore")

SEED, N_FOLDS = 42, 5
TARGET = "Will_Buy_EV"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/train.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Abrir desde notebook/ o desde la raiz del episodio.")
OUT = ROOT / "models" / "7_wide"
OUT.mkdir(parents=True, exist_ok=True)
print("lightgbm", lgb.__version__, "| salida:", OUT)

lightgbm 4.5.0 | salida: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 9 - Predicting Electric Vehicle Purchases\models\7_wide


## 1. Features

Tres bloques, separables para la ablación:

- **`BASE`** — las 13 crudas.
- **`ART`** — los artefactos medidos del generador: los dos flags, los dos dígitos de distancia y las
  frecuencias transductivas.
- **`TE`** — el encoding ancho, que se calcula dentro del fold.

`TargetEncoder.fit_transform` usa validación cruzada interna, así que las filas de entrenamiento
reciben una codificación que no vio su propia etiqueta; `transform` aplica el ajuste completo a
validación y test. Es la forma correcta de usarlo y la que evita la fuga.

Las frecuencias sí se calculan sobre train+test juntos: sólo usan la distribución de las features, no
el target, así que no hay nada que filtrar.

In [2]:
train = pd.read_csv(ROOT / "data/train.csv")
test  = pd.read_csv(ROOT / "data/test.csv")
sample = pd.read_csv(ROOT / "data/sample_submission.csv")
assert test["id"].equals(sample["id"])
y = train[TARGET].eq("Yes").to_numpy(np.int8)

CAT = ["Gender","City_Type","Current_Car_Type",
       "Home_Charging_Possible","Subsidy_Available","Range_Anxiety_Level"]
NUM = ["Age","Annual_Income_USD","Daily_Commute_km","Number_of_Cars_Owned",
       "Charging_Stations_Near_Home","Charging_Stations_Near_Work",
       "Environmental_Concern_Level"]

def construir(df, freq_maps):
    d = df.copy()
    inc, com = d["Annual_Income_USD"], d["Daily_Commute_km"]
    d["is_cliff"] = (inc >= 170537).astype(np.int8)     # tasa medida 1,0000
    d["is_30k"]   = (inc == 30000).astype(np.int8)      # mode collapse
    d["com_unit"] = (np.floor(com) % 10).astype(np.int8)
    d["com_dec"]  = (np.round(com * 10) % 10).astype(np.int8)
    d["inc_100"]  = np.floor(inc / 100) * 100
    d["inc_1000"] = np.floor(inc / 1000) * 1000
    d["com_1"]    = np.floor(com)
    for col, m in freq_maps.items():
        d[f"fe_{col}"] = d[col].map(m).fillna(0.0)
    return d

CLAVES = ["Annual_Income_USD","inc_100","inc_1000","Daily_Commute_km","com_1"]
FREQ_COLS = CLAVES + ["com_unit","com_dec"]
TE_COLS = CLAVES + CAT + ["Age","Environmental_Concern_Level",
                          "Charging_Stations_Near_Home","Charging_Stations_Near_Work"]

# Frecuencias transductivas: train+test, sin etiquetas
ambos = construir(pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True), {})
freq_maps = {c: ambos[c].value_counts(normalize=True).to_dict() for c in FREQ_COLS}

TR = construir(train, freq_maps)
TE_ = construir(test, freq_maps)
for c in CAT:
    dt = pd.CategoricalDtype(sorted(set(TR[c]) | set(TE_[c])))
    TR[c] = TR[c].astype(dt); TE_[c] = TE_[c].astype(dt)

BASE = NUM + CAT
ART  = ["is_cliff","is_30k","com_unit","com_dec"] + [f"fe_{c}" for c in FREQ_COLS]
print(f"BASE: {len(BASE)} | ART: {len(ART)} | claves a codificar: {len(TE_COLS)}")
print(f"cliff en train: {TR.is_cliff.sum():,} filas | en test: {TE_.is_cliff.sum():,}")

BASE: 13 | ART: 11 | claves a codificar: 15
cliff en train: 393 filas | en test: 156


## 2. Validación

`StratifiedKFold(5, shuffle=True, random_state=42)` — **los mismos folds que todos los notebooks
anteriores**, lo que permite comparación pareada y ensamble con los OOF guardados.

Cinco variantes comparten el encoding de cada fold. Cada una cambia una sola cosa:

| Variante | Qué aísla |
|---|---|
| `base` | LightGBM sobre las crudas, sin nada |
| `base_art` | Lo que aportan los artefactos del generador |
| `wide_te` | Lo que aporta el encoding ancho, sin artefactos |
| `full` | Los dos juntos |
| `full_leaves128` | Capacidad, con features idénticas |

In [3]:
PARAMS = dict(n_estimators=3000, learning_rate=0.05, num_leaves=64,
              min_child_samples=40, subsample=0.9, subsample_freq=1,
              colsample_bytree=0.7, reg_lambda=5.0,
              random_state=SEED, n_jobs=4, verbose=-1)
SMOOTHS = [("auto","auto"), ("10", 10.0), ("100", 100.0)]

def te_cols(tag):
    return [f"te{tag}_{c}" for c in TE_COLS]
TE_ALL = [c for tag,_ in SMOOTHS for c in te_cols(tag)]

VARIANTES = {
    "base":           (BASE,              64),
    "base_art":       (BASE+ART,          64),
    "wide_te":        (BASE+TE_ALL,       64),
    "full":           (BASE+ART+TE_ALL,   64),
    "full_leaves128": (BASE+ART+TE_ALL,  128),
}
for n,(f,l) in VARIANTES.items():
    print(f"  {n:16s} leaves={l:3d}  features={len(f)}")

  base             leaves= 64  features=13
  base_art         leaves= 64  features=24
  wide_te          leaves= 64  features=58
  full             leaves= 64  features=69
  full_leaves128   leaves=128  features=69


In [4]:
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
oof  = {n: np.zeros(len(TR), dtype=np.float32) for n in VARIANTES}
pred = {n: np.zeros(len(TE_), dtype=np.float64) for n in VARIANTES}
fold_ids = np.full(len(TR), -1, dtype=np.int8)
filas = []
t_ini = time.perf_counter()

for fold, (itr, iva) in enumerate(skf.split(TR, y), 1):
    fold_ids[iva] = fold
    Xtr, Xva, Xte = TR.iloc[itr].copy(), TR.iloc[iva].copy(), TE_.copy()
    t0 = time.perf_counter()
    # fit_transform hace cross-fitting interno: la fila no ve su propia etiqueta
    for tag, smooth in SMOOTHS:
        enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=SEED)
        a = enc.fit_transform(Xtr[TE_COLS].astype(str), y[itr])
        b = enc.transform(Xva[TE_COLS].astype(str))
        c = enc.transform(Xte[TE_COLS].astype(str))
        for i, col in enumerate(TE_COLS):
            Xtr[f"te{tag}_{col}"] = a[:, i]
            Xva[f"te{tag}_{col}"] = b[:, i]
            Xte[f"te{tag}_{col}"] = c[:, i]
    print(f"Fold {fold}/{N_FOLDS}: encoding {time.perf_counter()-t0:.0f}s", flush=True)

    for nombre, (feats, leaves) in VARIANTES.items():
        m = lgb.LGBMClassifier(**{**PARAMS, "num_leaves": leaves})
        m.fit(Xtr[feats], y[itr], eval_set=[(Xva[feats], y[iva])], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        pv = m.predict_proba(Xva[feats])[:, 1]
        pt = m.predict_proba(Xte[feats])[:, 1]
        assert np.isfinite(pv).all() and np.isfinite(pt).all()
        oof[nombre][iva] = pv
        pred[nombre] += pt / N_FOLDS
        auc = roc_auc_score(y[iva], pv)
        filas.append({"model": nombre, "fold": fold, "auc": auc})
        print(f"  {nombre:16s} AUC={auc:.6f}  iter={m.best_iteration_}", flush=True)

assert (fold_ids > 0).all()
print(f"\nTiempo total: {(time.perf_counter()-t_ini)/60:.1f} min")

Fold 1/5: encoding 73s
  base             AUC=0.940658  iter=496
  base_art         AUC=0.942898  iter=716
  wide_te          AUC=0.944887  iter=170
  full             AUC=0.944904  iter=185
  full_leaves128   AUC=0.944792  iter=159
Fold 2/5: encoding 81s
  base             AUC=0.941556  iter=434
  base_art         AUC=0.943840  iter=641
  wide_te          AUC=0.945555  iter=217
  full             AUC=0.945515  iter=259
  full_leaves128   AUC=0.945488  iter=185
Fold 3/5: encoding 87s
  base             AUC=0.942885  iter=506
  base_art         AUC=0.944913  iter=650
  wide_te          AUC=0.946697  iter=267
  full             AUC=0.946686  iter=302
  full_leaves128   AUC=0.946533  iter=225
Fold 4/5: encoding 67s
  base             AUC=0.942262  iter=306
  base_art         AUC=0.944256  iter=628
  wide_te          AUC=0.946158  iter=159
  full             AUC=0.946144  iter=163
  full_leaves128   AUC=0.946047  iter=128
Fold 5/5: encoding 94s
  base             AUC=0.941906  iter=356
  b

## 3. Comparación pareada contra los modelos previos

Se reporta el **desvío de la diferencia pareada**, no la dispersión entre folds: los folds son los
mismos para todos los modelos y esa varianza se cancela al restar.

In [5]:
for nombre, archivo in [("nb5_hybrid","5_hybrid_oof.npy"),
                        ("nb6_newton","6_hierarchical/predictions.npz")]:
    ruta = ROOT / "models" / archivo
    if not ruta.exists():
        print("No disponible:", archivo); continue
    p = np.load(ruta)["newton_oof"] if ruta.suffix == ".npz" else np.load(ruta)
    oof[nombre] = p.astype(np.float32)
    for f in range(1, N_FOLDS+1):
        m = fold_ids == f
        filas.append({"model": nombre, "fold": f, "auc": roc_auc_score(y[m], p[m])})

fm = pd.DataFrame(filas)
paired = fm.pivot(index="fold", columns="model", values="auc")
resumen = pd.DataFrame({n: {"auc_oof": roc_auc_score(y, p),
                            "media_fold": paired[n].mean(),
                            "sd_fold": paired[n].std(ddof=1)}
                        for n, p in oof.items()}).T
print(resumen.sort_values("auc_oof", ascending=False).round(6).to_string())

print("\nAblaciones (cada una aisla un solo cambio):")
ABL = [("base_art","base","artefactos del generador"),
       ("wide_te","base","encoding ancho"),
       ("full","wide_te","artefactos sobre el encoding"),
       ("full","base_art","encoding sobre los artefactos"),
       ("full_leaves128","full","capacidad 64 -> 128 hojas")]
if "nb6_newton" in paired:
    ABL.append(("full","nb6_newton","este modelo vs el notebook 6"))
for nuevo, ctrl, que in ABL:
    d = paired[nuevo] - paired[ctrl]
    t = ttest_rel(paired[nuevo], paired[ctrl])
    print(f"  {que:34s} {d.mean():+.6f} | folds {int((d>0).sum())}/{N_FOLDS}"
          f" | sd pareado {d.std(ddof=1):.6f} | t={t.statistic:6.2f}")

                 auc_oof  media_fold   sd_fold
wide_te         0.945838    0.945844  0.000677
full            0.945818    0.945825  0.000669
full_leaves128  0.945721    0.945732  0.000649
nb6_newton      0.945468    0.945470  0.000722
nb5_hybrid      0.945159    0.945162  0.000708
base_art        0.943954    0.943959  0.000732
base            0.941844    0.941853  0.000829

Ablaciones (cada una aisla un solo cambio):
  artefactos del generador           +0.002106 | folds 5/5 | sd pareado 0.000144 | t= 32.67
  encoding ancho                     +0.003990 | folds 5/5 | sd pareado 0.000157 | t= 56.89
  artefactos sobre el encoding       -0.000019 | folds 1/5 | sd pareado 0.000025 | t= -1.64
  encoding sobre los artefactos      +0.001866 | folds 5/5 | sd pareado 0.000141 | t= 29.52
  capacidad 64 -> 128 hojas          -0.000093 | folds 0/5 | sd pareado 0.000047 | t= -4.48
  este modelo vs el notebook 6       +0.000355 | folds 5/5 | sd pareado 0.000069 | t= 11.56


## 4. Diversidad y ensamble

Es la pregunta central del notebook. Todos los ensambles del episodio fracasaron porque los modelos
ordenaban casi igual: Spearman 0,9889 con el logit, 0,9945 con el XGBoost tuneado, 0,9984 entre los
dos híbridos. Si este LightGBM baja de eso, el ensamble tiene por dónde ganar aunque el modelo
individual no gane.

> El AUC del mejor peso está **sesgado al alza**: los pesos se eligen sobre el mismo OOF con el que se
> reporta. Sirve para decidir si vale la pena mezclar, no como estimación de lo que dará el
> leaderboard. Por eso se reporta también una mezcla 50/50 fijada de antemano.

In [6]:
if "nb6_newton" in oof:
    r_new = rankdata(oof["full"]) / len(y)
    r_old = rankdata(oof["nb6_newton"]) / len(y)
    print(f"Spearman full vs notebook 6: {np.corrcoef(r_new, r_old)[0,1]:.5f}")
    print("  referencias del episodio: logit 0,9889 | xgb tuneado 0,9945 | hibridos 0,9984\n")

    mejor = max(((w, roc_auc_score(y, w*r_new + (1-w)*r_old))
                 for w in np.round(np.arange(0, 1.01, 0.05), 2)), key=lambda t: t[1])
    auc_5050 = roc_auc_score(y, 0.5*r_new + 0.5*r_old)
    solo = max(roc_auc_score(y, oof["full"]), roc_auc_score(y, oof["nb6_newton"]))
    print(f"  mezcla 50/50 (prefijada):  {auc_5050:.6f}   ({auc_5050-solo:+.6f} vs el mejor solo)")
    print(f"  mejor peso w={mejor[0]:.2f} (sesgado): {mejor[1]:.6f}   ({mejor[1]-solo:+.6f})")
    print(f"  mejor modelo individual:   {solo:.6f}")

Spearman full vs notebook 6: 0.99250
  referencias del episodio: logit 0,9889 | xgb tuneado 0,9945 | hibridos 0,9984

  mezcla 50/50 (prefijada):  0.945991   (+0.000173 vs el mejor solo)
  mejor peso w=0.60 (sesgado): 0.946010   (+0.000192)
  mejor modelo individual:   0.945818


## 5. Artefactos y submissions

In [7]:
np.save(ROOT/"models/7_wide_oof.npy",  oof["full"])
np.save(ROOT/"models/7_wide_test.npy", pred["full"])
payload = {"train_ids": train["id"].to_numpy(), "test_ids": test["id"].to_numpy(),
           "fold_ids": fold_ids, "y": y}
for n in VARIANTES:
    payload[n+"_oof"] = oof[n]; payload[n+"_test"] = pred[n]
np.savez_compressed(OUT/"predictions.npz", **payload)
resumen.to_csv(OUT/"metrics.csv"); fm.to_csv(OUT/"fold_metrics.csv", index=False)
(OUT/"config.json").write_text(json.dumps(
    {"seed": SEED, "folds": N_FOLDS, "params": PARAMS,
     "variantes": {k: [v[0], v[1]] for k, v in VARIANTES.items()},
     "te_cols": TE_COLS, "smooths": [s[0] for s in SMOOTHS]}, indent=2), encoding="utf-8")

dest = ROOT/"submissions"; dest.mkdir(exist_ok=True)
for n in VARIANTES:
    sub = pd.DataFrame({"id": test["id"], TARGET: pred[n]})
    assert sub["id"].equals(sample["id"]) and sub[TARGET].between(0,1).all()
    sub.to_csv(dest/f"7_{n}_submission.csv", index=False)
if "nb6_newton" in oof:
    mezcla = 0.5*(rankdata(pred["full"])/len(pred["full"])) + \
             0.5*(rankdata(np.load(ROOT/"models/6_hierarchical/predictions.npz")["newton_test"])
                  / len(pred["full"]))
    pd.DataFrame({"id": test["id"], TARGET: mezcla}).to_csv(
        dest/"7_blend50_nb6_submission.csv", index=False)
print("CSVs escritos en", dest)
print("Revisar metrics.csv antes de elegir un envio. No se envia nada automaticamente.")

CSVs escritos en c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 9 - Predicting Electric Vehicle Purchases\submissions
Revisar metrics.csv antes de elegir un envio. No se envia nada automaticamente.


## 6. Resultados

_Pendiente: se completa después de ejecutar._